Goal: Load pre-trained Resnet-50 Model and apply Grad-CAM

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Import library

In [2]:
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from torchvision import models, transforms

# Grad-CAM
!pip install grad-cam -q

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 57.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


Config

In [3]:
class Config:
    resnet_model_path = "/content/drive/MyDrive/vit_checkpoints/best_model.pth"
    vit_model_path = "/content/drive/MyDrive/vit_checkpoints/best_model_vit.pth"
    folder_path = "/content/drive/MyDrive/NEU/DS5500-Data Capstone/DS5500_AIGI-Detection/AIGI-data/GRAD_CAM/"
    image_size = 224
    device = "cuda" if torch.cuda.is_available() else "cpu"

cfg = Config()
print(cfg.device)

cpu


Rebuild Model (ResNet50)

In [4]:
def build_model():
    model = models.resnet50(pretrained=False)
    model.fc = nn.Linear(model.fc.in_features, 2)  # Changed from 1 to 2 to match the checkpoint
    return model

def load_model(path, device):
    model = build_model()
    state_dict = torch.load(path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model

model = load_model(cfg.resnet_model_path, cfg.device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Image Preprocessing

In [5]:
transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def load_image(path):
    img = Image.open(path).convert("RGB")
    img_resized = img.resize((cfg.image_size, cfg.image_size))

    input_tensor = transform(img).unsqueeze(0)

    return img, img_resized, input_tensor

In [ ]:
# This cell is no longer needed as prediction and Grad-CAM will be done in a loop below.
# Keeping it blank for now, or removing its content if it's already empty.

Grad-CAM Setup (will be done once)

In [6]:
# Target last convolutional layer
target_layer = model.layer4[-1]

cam = GradCAM(
    model=model,
    target_layers=[target_layer],
)

Generate Grad-CAM and display for all images in the folder

In [7]:
import os

image_files = [f for f in os.listdir(cfg.folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]

if not image_files:
    print(f"No image files found in the directory: {cfg.folder_path}")
else:
    for img_file in image_files:
        img_path = os.path.join(cfg.folder_path, img_file)
        print(f"\nProcessing image: {img_file}")

        try:
            img, img_resized, input_tensor = load_image(img_path)
            input_tensor = input_tensor.to(cfg.device)

            # Model Prediction
            with torch.no_grad():
                output = model(input_tensor)
                # Apply softmax to get probabilities for each class, then select the probability of the second class (index 1)
                prob = torch.softmax(output, dim=1)[0, 1].item()

            print(f"Prediction (AI probability): {prob:.4f}")

            # Generate Grad-CAM
            grayscale_cam = cam(input_tensor=input_tensor)[0]
            rgb_img = np.array(img_resized) / 255.0
            visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

            # Visualize Results
            plt.figure(figsize=(10,5))

            plt.subplot(1,2,1)
            plt.title(f"Original Image: {img_file}")
            plt.imshow(img_resized)
            plt.axis("off")

            plt.subplot(1,2,2)
            plt.title("Grad-CAM")
            plt.imshow(visualization)
            plt.axis("off")

            plt.show()
        except Exception as e:
            print(f"Error processing {img_file}: {e}")

Output hidden; open in https://colab.research.google.com to view.

## Rebuild and Load ViT Model

In [8]:
import torchvision.models as models
import torch.nn as nn

def build_vit_model():
    # Using a pre-trained ViT-B/16 model, setting pretrained=False as weights will be loaded from checkpoint
    vit_model = models.vit_b_16(pretrained=False)
    # Adjust the classification head for 2 output classes (e.g., AI vs Human)
    vit_model.heads.head = nn.Linear(vit_model.heads.head.in_features, 2)
    return vit_model

def load_vit_model(path, device):
    vit_model = build_vit_model()
    # Load the state dictionary from your checkpoint
    state_dict = torch.load(path, map_location=device)
    vit_model.load_state_dict(state_dict)
    vit_model.to(device)
    vit_model.eval() # Set model to evaluation mode
    return vit_model

# Load the ViT model using your configured path
vit_model = load_vit_model(cfg.vit_model_path, cfg.device)
print(f"ViT model loaded successfully on {cfg.device}")

ViT model loaded successfully on cpu


## Grad-CAM Setup for ViT Model

In [15]:
from pytorch_grad_cam import GradCAM

# Define a reshape_transform for ViT models
# This function reshapes the output of the attention layer to match image dimensions
def reshape_transform(tensor, height=14, width=14):
    result = tensor[:, 1:, :].reshape(tensor.size(0), height, width, tensor.size(2))
    # Bring the channels to the first dimension, like in CNNs.
    result = result.permute(0, 3, 1, 2)
    return result

# For ViT models, a good target layer for Grad-CAM is often the last encoder block
# Accessing the last layer of the encoder blocks: model.encoder.layers[-1]
vit_target_layer = vit_model.encoder.layers[-1].ln_1

# Initialize GradCAM for the ViT model with the reshape_transform
vit_cam = GradCAM(
    model=vit_model,
    target_layers=[vit_target_layer],
    reshape_transform=reshape_transform
)
print("Grad-CAM for ViT initialized with target layer: model.encoder.layers[-1].ln_1 and reshape_transform.")

Grad-CAM for ViT initialized with target layer: model.encoder.layers[-1] and reshape_transform.


## Generate ViT Predictions and Grad-CAM for all images in the folder

In [16]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget # Import this for target category

image_files = [f for f in os.listdir(cfg.folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]

if not image_files:
    print(f"No image files found in the directory: {cfg.folder_path}")
else:
    for img_file in image_files:
        img_path = os.path.join(cfg.folder_path, img_file)
        print(f"\nProcessing image with ViT: {img_file}")

        try:
            # Load and preprocess the image using the existing load_image function
            img, img_resized, input_tensor = load_image(img_path)
            input_tensor = input_tensor.to(cfg.device)

            # ViT Model Prediction
            with torch.no_grad():
                output = vit_model(input_tensor)
                # Apply softmax to get probabilities for each class, then select the probability of the second class (index 1)
                prob = torch.softmax(output, dim=1)[0, 1].item() # Assuming index 1 corresponds to 'AI' probability

            print(f"ViT Prediction (AI probability): {prob:.4f}")

            # Generate Grad-CAM for ViT
            # For ViT, the default output of reshape_transform is 14x14, so we upsample it to the original image size

            # Specify the target category for Grad-CAM. Assuming index 1 is the 'AI' class
            targets = [ClassifierOutputTarget(1)]

            vit_grayscale_cam = vit_cam(input_tensor=input_tensor, targets=targets)[0]
            rgb_img = np.array(img_resized) / 255.0 # Normalize for visualization
            vit_visualization = show_cam_on_image(rgb_img, vit_grayscale_cam, use_rgb=True)

            # Visualize Results
            plt.figure(figsize=(12, 6)) # Increased figure size for better display

            plt.subplot(1,2,1)
            plt.title(f"Original Image: {img_file}")
            plt.imshow(img_resized)
            plt.axis("off")

            plt.subplot(1,2,2)
            plt.title("ViT Grad-CAM")
            plt.imshow(vit_visualization)
            plt.axis("off")

            plt.show()
        except Exception as e:
            print(f"Error processing {img_file} with ViT: {e}")

Output hidden; open in https://colab.research.google.com to view.